In [5]:
# ======================================================
# WEEK 7 BBO STRATEGY
# Bayesian Optimization + Hyperparameter Tuning
# Week 1 to Week 6 data included
# ======================================================

import numpy as np
import warnings

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, RBF, WhiteKernel, ConstantKernel
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import mean_squared_error

warnings.filterwarnings("ignore")
np.random.seed(42)

# ======================================================
# WEEK 1–6 INPUT DATA
# ======================================================

inputs_by_week = [
    [
        np.array([0.211325, 0.788675]),
        np.array([0.723607, 0.276393]),
        np.array([0.166667, 0.500000, 0.833333]),
        np.array([0.125000, 0.375000, 0.625000, 0.875000]),
        np.array([0.875000, 0.625000, 0.375000, 0.125000]),
        np.array([0.150000, 0.350000, 0.550000, 0.750000, 0.950000]),
        np.array([0.900000, 0.100000, 0.700000, 0.300000, 0.500000, 0.800000]),
        np.array([0.111111, 0.222222, 0.333333, 0.444444, 0.555555, 0.666667, 0.777778, 0.888889])
    ],
    [
        np.array([0.654299, 0.353479]),
        np.array([0.754299, 0.253479]),
        np.array([0.304299, 0.553479, 0.709691]),
        np.array([0.804299, 0.603479, 0.409691, 0.205016]),
        np.array([0.854299, 0.653479, 0.359691, 0.155016]),
        np.array([0.904299, 0.703479, 0.509691, 0.305016, 0.103275]),
        np.array([0.884299, 0.123479, 0.729691, 0.285016, 0.523275, 0.787492]),
        np.array([0.104299, 0.243479, 0.319691, 0.465016, 0.573275, 0.647492, 0.794889, 0.860861])
    ],
    [
        np.array([0.582812, 0.421077]),
        np.array([0.712865, 0.284413]),
        np.array([0.118496, 0.481282, 0.876608]),
        np.array([1.000000, 0.683447, 0.334333, 0.000000]),
        np.array([0.882245, 0.615032, 0.380358, 0.114494]),
        np.array([0.000000, 0.226282, 0.564108, 0.905744, 1.000000]),
        np.array([0.905495, 0.091782, 0.689608, 0.305244, 0.491854, 0.804378]),
        np.array([0.113495, 0.214782, 0.338108, 0.437244, 0.549353, 0.673378, 0.771789, 0.898699])
    ],
    [
        np.array([0.183704, 0.127166]),
        np.array([0.255353, 0.749841]),
        np.array([0.094906, 0.770362, 0.859589]),
        np.array([0.820856, 0.226975, 0.784625, 0.113609]),
        np.array([0.917860, 0.273128, 0.475575, 0.052446]),
        np.array([0.052509, 0.789636, 0.354234, 0.228337, 0.889991]),
        np.array([0.833796, 0.096195, 0.232612, 0.789158, 0.185769, 0.681185]),
        np.array([0.448564, 0.335191, 0.751694, 0.277261, 0.167614, 0.717472, 0.304117, 0.767059])
    ],
    [
        np.array([0.582812, 0.421077]),
        np.array([0.684812, 0.281383]),
        np.array([0.171685, 0.508728, 0.819757]),
        np.array([0.734042, 0.687625, 0.466427, 0.303264]),
        np.array([0.568707, 0.909615, 0.957143, 0.011420]),
        np.array([0.744968, 0.084644, 0.801527, 0.169728, 0.990511]),
        np.array([1.000000, 0.229371, 0.690054, 0.244152, 0.521424, 1.000000]),
        np.array([0.022620, 0.000000, 0.288709, 0.762817, 0.551656, 0.961269, 0.843087, 1.000000])
    ],
    [
        np.array([0.582812, 0.421077]),
        np.array([0.996744, 0.999561]),
        np.array([0.166667, 0.500000, 0.833333]),
        np.array([0.743137, 0.718436, 0.581810, 0.365984]),
        np.array([0.572707, 0.914615, 0.952143, 0.016420]),
        np.array([0.236209, 0.379873, 0.584786, 0.843391, 1.000000]),
        np.array([0.024461, 0.059677, 0.995330, 0.299568, 0.038576, 0.691027]),
        np.array([0.148189, 0.189483, 0.270757, 0.392507, 0.446351, 0.670593, 0.665417, 1.000000])
    ]
]

# ======================================================
# WEEK 1–6 OUTPUT DATA
# ======================================================

outputs_by_week = [
    np.array([
        1.1327056288856873e-125,
        0.5675786892822564,
        -0.032385408076210126,
        -17.20744048260943,
        60.223125,
        -1.3287857969718009,
        0.34356041660314957,
        9.0501517903694
    ]),
    np.array([
        1.1867858443665185e-41,
        0.2715258567299176,
        -0.1198581070659559,
        -13.082213230390916,
        50.179390256321376,
        -1.5080002951000964,
        0.31639485635485903,
        9.0219949134074
    ]),
    np.array([
        7.99676981308551e-19,
        0.5213723465552891,
        -0.04726977098841539,
        -25.67625344929702,
        64.78245026474816,
        -1.7372122723701597,
        0.3507828450585503,
        9.0587238074074
    ]),
    np.array([
        -2.410121917336285e-100,
        0.0244939290195959,
        -0.04207093453964322,
        -20.330739763644917,
        61.278397876784794,
        -2.4624737429462145,
        0.0634803557047261,
        8.275283250689899
    ]),
    np.array([
        7.99676981308551e-19,
        0.4258127251524913,
        -0.06232295859499999,
        -12.496845976106417,
        942.2521944777988,
        -2.280900502246122,
        0.21828066675598462,
        8.427686115001
    ]),
    np.array([
        7.99676981308551e-19,
        -0.004122974640885927,
        -0.032385408076210126,
        -14.192389833871584,
        943.6841142765069,
        -1.2257941580841207,
        0.4410410448155631,
        9.3346877420455
    ])
]

# ======================================================
# HELPER FUNCTIONS
# ======================================================

def get_history(function_index):
    X = np.vstack([week[function_index] for week in inputs_by_week])
    y = np.array([week[function_index] for week in outputs_by_week])
    return X, y


def tune_gaussian_process(X, y):
    """
    Bayesian-style hyperparameter tuning.
    We test several kernels and noise levels,
    then choose the one with the lowest leave-one-out error.
    """

    kernels = [
        ConstantKernel(1.0) * Matern(length_scale=1.0, nu=1.5) + WhiteKernel(noise_level=1e-5),
        ConstantKernel(1.0) * Matern(length_scale=1.0, nu=2.5) + WhiteKernel(noise_level=1e-5),
        ConstantKernel(1.0) * RBF(length_scale=1.0) + WhiteKernel(noise_level=1e-5)
    ]

    alphas = [1e-8, 1e-6, 1e-4, 1e-3]

    best_model = None
    best_error = float("inf")
    best_details = None

    loo = LeaveOneOut()

    for kernel in kernels:
        for alpha in alphas:
            errors = []

            for train_idx, test_idx in loo.split(X):
                try:
                    model = GaussianProcessRegressor(
                        kernel=kernel,
                        alpha=alpha,
                        normalize_y=True,
                        n_restarts_optimizer=8,
                        random_state=42
                    )

                    model.fit(X[train_idx], y[train_idx])
                    pred = model.predict(X[test_idx])
                    errors.append(mean_squared_error(y[test_idx], pred))
                except:
                    errors.append(1e9)

            avg_error = np.mean(errors)

            if avg_error < best_error:
                best_error = avg_error
                best_model = GaussianProcessRegressor(
                    kernel=kernel,
                    alpha=alpha,
                    normalize_y=True,
                    n_restarts_optimizer=15,
                    random_state=42
                )
                best_details = {
                    "kernel": str(kernel),
                    "alpha": alpha,
                    "loo_mse": avg_error
                }

    best_model.fit(X, y)
    return best_model, best_details


def generate_candidates(X, y, function_index, n_local=8000, n_global=2500):
    """
    Candidate strategy:
    - Strong exploitation around best point
    - Some search around second-best point
    - Some interpolation between top points
    - Controlled global exploration
    """

    dim = X.shape[1]
    order = np.argsort(y)[::-1]

    best_x = X[order[0]]
    second_x = X[order[1]]

    # Function 5 gets stronger exploitation due to Week 5 and Week 6 gains
    if function_index == 4:
        local_radius = 0.018
        second_radius = 0.035
        global_count = 500
    else:
        local_radius = 0.050
        second_radius = 0.080
        global_count = n_global

    local_best = np.clip(
        best_x + np.random.normal(0, local_radius, size=(n_local, dim)),
        0, 0.995
    )

    local_second = np.clip(
        second_x + np.random.normal(0, second_radius, size=(n_local // 2, dim)),
        0, 0.995
    )

    mix = np.random.uniform(0, 1, size=(1500, 1))
    interpolation = np.clip(
        mix * best_x + (1 - mix) * second_x + np.random.normal(0, 0.020, size=(1500, dim)),
        0, 0.995
    )

    global_search = np.random.uniform(0.001, 0.995, size=(global_count, dim))

    candidates = np.vstack([
        local_best,
        local_second,
        interpolation,
        global_search,
        X
    ])

    return candidates, best_x


def acquisition_ucb(mu, sigma, candidates, best_x, beta=2.0, distance_penalty=0.04):
    """
    Upper Confidence Bound:
    score = predicted mean + beta * uncertainty - distance penalty
    """

    distance = np.linalg.norm(candidates - best_x, axis=1)
    score = mu + beta * sigma - distance_penalty * distance
    return score


def propose_week7(function_index):
    X, y = get_history(function_index)

    best_y = np.max(y)
    best_x = X[np.argmax(y)]

    model, details = tune_gaussian_process(X, y)
    candidates, best_x = generate_candidates(X, y, function_index)

    mu, sigma = model.predict(candidates, return_std=True)

    # Function-specific acquisition logic
    if function_index == 4:
        # Function 5: exploit heavily
        beta = 0.35
        distance_penalty = 0.12
    elif function_index in [6, 7]:
        # Functions 7 and 8 improved in Week 6
        beta = 1.2
        distance_penalty = 0.05
    else:
        # Other uncertain functions
        beta = 2.0
        distance_penalty = 0.04

    score = acquisition_ucb(
        mu,
        sigma,
        candidates,
        best_x,
        beta=beta,
        distance_penalty=distance_penalty
    )

    selected = candidates[np.argmax(score)]

    return np.round(selected, 6), best_y, best_x, details


# ======================================================
# GENERATE WEEK 7 INPUTS
# ======================================================

week7_inputs = []

print("WEEK 7 BAYESIAN OPTIMIZATION INPUTS")
print("===================================")

for i in range(8):
    x7, best_y, best_x, details = propose_week7(i)
    week7_inputs.append(x7)

    print(f"\nFunction {i+1}")
    print(f"Best output so far: {best_y}")
    print(f"Best input so far:  {np.round(best_x, 6)}")
    print(f"Week 7 input:       {x7}")
    print(f"Best GP setup:      {details}")

print("\nPORTAL-READY WEEK 7 INPUTS")
print("==========================")

for i, arr in enumerate(week7_inputs, start=1):
    formatted = "-".join(f"{x:.6f}" for x in arr)
    print(f"Function {i}: {formatted}")

WEEK 7 BAYESIAN OPTIMIZATION INPUTS

Function 1
Best output so far: 7.99676981308551e-19
Best input so far:  [0.582812 0.421077]
Week 7 input:       [0.582812 0.421077]
Best GP setup:      {'kernel': '1**2 * RBF(length_scale=1) + WhiteKernel(noise_level=1e-05)', 'alpha': 1e-08, 'loo_mse': np.float64(1.1510698940202231e-37)}

Function 2
Best output so far: 0.5675786892822564
Best input so far:  [0.723607 0.276393]
Week 7 input:       [0.744767 0.322712]
Best GP setup:      {'kernel': '1**2 * Matern(length_scale=1, nu=1.5) + WhiteKernel(noise_level=1e-05)', 'alpha': 0.001, 'loo_mse': np.float64(0.06537889071384707)}

Function 3
Best output so far: -0.032385408076210126
Best input so far:  [0.166667 0.5      0.833333]
Week 7 input:       [0.167431 0.503246 0.833129]
Best GP setup:      {'kernel': '1**2 * RBF(length_scale=1) + WhiteKernel(noise_level=1e-05)', 'alpha': 0.001, 'loo_mse': np.float64(0.0010773753659960314)}

Function 4
Best output so far: -12.496845976106417
Best input so far: